In [6]:
import pandas as pd
import json
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score,
    confusion_matrix,
    classification_report
)

# Functions

In [7]:
def get_summary_classification_metrics(

    y_true: "array-like",
    y_predict: "array-like"

) -> dict:

    summary = {
        "f1_score": f1_score(y_true, y_predict, average="macro"),
        "classification_report": classification_report(y_true, y_predict),
        "confusion_matrix": confusion_matrix(y_true, y_predict).tolist()
    }

    return summary

def get_balance_classes(

    df: pd.DataFrame,
    label_col_name:str,
    how="over"

) -> pd.DataFrame:

    # Arguments necessary to undersampling or oversampling
    how_dict_args = {
        "over": (max, True),
        "under": (min, False)
    }
    func_counts, bool_replace = how_dict_args[how]

    label_counts = df[label_col_name].value_counts()

    target_label_count =  func_counts(label_counts)
    target_label_name = label_counts[label_counts == target_label_count].index[0]

    label_count_no_target = label_counts[label_counts != target_label_count]

    sample_df = pd.DataFrame()

    for label, _ in label_count_no_target.items():

        tmp = df.query(f"{label_col_name} == {label}")
        tmp = tmp.sample(n=target_label_count, replace=bool_replace)

        sample_df = pd.concat([sample_df, tmp])

    tmp = df.query(f"{label_col_name} == {target_label_name}")
    sample_df = pd.concat([tmp, sample_df])

    sample_df = sample_df.sample(n=sample_df.shape[0]).reset_index(drop=True)
    return sample_df

In [8]:
print("Reading data")
loan_default_df = pd.read_parquet("../data/processed/processed_data.parquet")
loan_default_df.head()

Reading data


,remainder__Default,edu__Education,emp__EmploymentType,ohe__MaritalStatus_Married,ohe__MaritalStatus_Single,ohe__LoanPurpose_Business,ohe__LoanPurpose_Education,ohe__LoanPurpose_Home,ohe__LoanPurpose_Other,ohe__HasMortgage_Yes,...,ohe__HasCoSigner_Yes,standarization__Age,standarization__Income,standarization__LoanAmount,standarization__CreditScore,standarization__MonthsEmployed,standarization__NumCreditLines,standarization__InterestRate,standarization__LoanTerm,standarization__DTIRatio
0,0,1.0,3.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.834048,0.089339,-1.087133,-0.341742,0.591797,1.341967,0.262037,-0.001770,-0.261175
1,0,2.0,3.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,...,1.0,1.701452,-0.823238,-0.044426,-0.731990,-1.285180,-1.343438,-1.307999,1.412531,0.778237
2,1,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.166815,0.043507,0.022609,-0.776050,-0.967538,0.446832,1.157048,-0.708920,-0.824191
3,0,0.0,3.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,-0.767312,-1.303597,-1.168852,1.061890,-1.718328,0.446832,-0.967473,-0.708920,-1.170661
4,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.100942,-1.592957,-1.672324,0.369515,-1.487316,1.341967,-1.051851,0.705381,0.994782


In [9]:
# Ratio between clases
class_ratio = loan_default_df["remainder__Default"].value_counts()
class_ratio

remainder__Default
0    234511
1     30835
Name: count, dtype: int64

In [10]:
class_ratio / class_ratio.sum()

remainder__Default
0    0.883793
1    0.116207
Name: count, dtype: float64

# Splitting

In [11]:
print("Splitting data 80 / 20")

y = loan_default_df.pop("remainder__Default")
X = loan_default_df

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, stratify=y)

print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

Splitting data 80 / 20
(212276, 20)
(212276,)
(53070, 20)
(53070,)


# Modeling

## Oversampling

In [12]:
print("Running Baseline Model")
rfc = RandomForestClassifier(
    n_jobs=-1,
    max_depth=20,
    min_samples_split=10,
    class_weight={0:1, 1:25}
)
rfc.fit(X_train, y_train)

Running Baseline Model


RandomForestClassifier(class_weight={0: 1, 1: 25}, max_depth=20,
                       min_samples_split=10, n_jobs=-1)

In [13]:
y_hat_train = rfc.predict(X_train)
y_hat_test  = rfc.predict(X_test)

summ_train = get_summary_classification_metrics(y_train, y_hat_train)
summ_test  = get_summary_classification_metrics(y_test, y_hat_test)

print(summ_train["f1_score"])
print(summ_train["classification_report"])
print(summ_train["confusion_matrix"])

print(summ_test["f1_score"])
print(summ_test["classification_report"])
print(summ_test["confusion_matrix"])

#summary_models["baseline"] = {"train": summ_train, "test": summ_test}

0.889525125621786
              precision    recall  f1-score   support

           0       1.00      0.94      0.97    187608
           1       0.68      1.00      0.81     24668

    accuracy                           0.95    212276
   macro avg       0.84      0.97      0.89    212276
weighted avg       0.96      0.95      0.95    212276

[[176106, 11502], [12, 24656]]
0.6368632365221475
              precision    recall  f1-score   support

           0       0.92      0.90      0.91     46903
           1       0.34      0.39      0.36      6167

    accuracy                           0.84     53070
   macro avg       0.63      0.65      0.64     53070
weighted avg       0.85      0.84      0.85     53070

[[42231, 4672], [3752, 2415]]


In [14]:
evluation_dict = {
    "train": summ_train,
    "test": summ_test
}

with open("tmp/evaluation.json", "w") as oJ:
    json.dump(evluation_dict, oJ, indent=4)

In [ ]:
with open("../models/randomForestLoanDefault.joblib", "wb") as oJ:
    joblib.dump(rfc, oJ)

RandomForestClassifier(class_weight={0: 1, 1: 25}, max_depth=20,
                       min_samples_split=10, n_jobs=-1)